# Bian et al. (2021) DTNN: Panasonic to CMU eVTOL

This notebook runs the supervised DTNN/MDA extension on the CMU core-seven protocol. It uses the paper-aligned serial `CNN1 -> CNN2 -> two-layer BiLSTM -> FC` network, sequence-to-sequence SOC regression, and MMD at CNN2, BiLSTM2, and FC.

CMU `soc_dynamic` is a development reference, not laboratory ground truth; these runs are not formal-benchmark eligible. Add a Colab Secret named `GITHUB_TOKEN`, enable Notebook access, and select a GPU runtime before the formal run.

In [30]:
from pathlib import Path
import os, subprocess, sys, tempfile
from urllib.parse import urlunsplit

from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = Path('/content/battery-soc-transfer-benchmark')
REPO_URL = urlunsplit(('https', 'github.com', '/zhangcastle/battery-soc-transfer-benchmark.git', '', ''))
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Add the GITHUB_TOKEN Colab Secret and enable Notebook access')
if REPO.exists() and not (REPO / '.git').exists():
    raise RuntimeError(f'Remove incomplete checkout and rerun: {REPO}')
with tempfile.TemporaryDirectory() as temp_dir:
    askpass = Path(temp_dir) / 'git-askpass.sh'
    askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token" ;;\n*) printf "%s" "$GITHUB_TOKEN" ;;\nesac\n')
    askpass.chmod(0o700)
    git_env = os.environ.copy()
    git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
    command = (['git', '-C', str(REPO), 'pull', '--ff-only'] if (REPO / '.git').exists() else ['git', 'clone', '--depth', '1', REPO_URL, str(REPO)])
    result = subprocess.run(command, env=git_env, text=True, capture_output=True)
del github_token
if result.returncode:
    raise RuntimeError('Authenticated GitHub setup failed; check token permission')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('Repository ready:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository ready: 0fc14046449dd63d9c1095923f42bcf11925992e


## Protocol

- Source: Panasonic NCR18650PF at 25, 10, 0, -10, and -20 C; Cycle 1-4, NN, US06, and HWFET.
- Inputs: current and voltage, normalized with the combined Panasonic source ranges; target data do not fit a scaler.
- Windows: 20 steps, non-overlapping; every step has an SOC target. The paper leaves M unspecified, and 20 keeps the short-cruise CMU condition eligible.
- CMU adaptation: chronological complete-mission prefix matching 8,778 seconds. Each mission starts at 70% SOC; its first 30% is labelled for adaptation.
- Held-out test missions also start at 70% SOC and are evaluated on the remaining 70%. Every cell and repeat starts from its source-pretrained state.
- Objective: source MSE + `5 * target MSE + 0.1 * adaptive multiscale MMD`.
- `formal` keeps the validated screening model and epoch counts, but repeats five seeds for uncertainty reporting.

In [31]:
import pandas as pd
import torch
from IPython.display import display
from battery_benchmark.utils.config import load_yaml

PATHS = load_yaml(REPO / 'configs/paths/colab_drive.yaml')
PANASONIC_ROOT = Path(PATHS['raw_data_root']) / 'Panasonic_18650PF'
CMU_BASE_ROOT = REPO / 'outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1'
CMU_CONDITION_ROOT = REPO / 'outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1_extension4'
CMU_MANIFEST = REPO / 'data/manifests/cmu_evtol/cmu_evtol_supervised_budget_manifest_v0_1.csv'
RUN_MODE = 'formal'  # screening | formal
RUN_EXPERIMENT = True
SETTINGS = {
    'screening': dict(source_epochs=20, transfer_epochs=5, repeats=1, batch_size=64, lightweight=True),
    'formal': dict(source_epochs=20, transfer_epochs=5, repeats=5, batch_size=64, lightweight=True),
}
setting = SETTINGS[RUN_MODE]
INITIAL_SOC = 0.70
RESULT_ROOT = Path(PATHS['output_root']) / 'benchmarks/panasonic_to_cmu_evtol/bian_2021/initial_soc_70_v2' / RUN_MODE
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if RUN_MODE == 'formal' and DEVICE != 'cuda':
    raise RuntimeError('Select a Colab GPU runtime for the formal run')
print('mode:', RUN_MODE, setting)
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
print('output:', RESULT_ROOT)

mode: formal {'source_epochs': 20, 'transfer_epochs': 5, 'repeats': 5, 'batch_size': 64, 'lightweight': True}
device: cuda NVIDIA L4
output: /content/drive/MyDrive/battery_soc_benchmark/outputs/benchmarks/panasonic_to_cmu_evtol/bian_2021/initial_soc_70_v2/formal


In [32]:
required = [PANASONIC_ROOT, CMU_BASE_ROOT, CMU_CONDITION_ROOT, CMU_MANIFEST, REPO / 'scripts/benchmarks/run_bian_cmu.py']
for path in required:
    assert path.exists(), f'Missing required input: {path}'
cmu_files = sorted(CMU_BASE_ROOT.glob('*_dynamic_soc.parquet')) + sorted(CMU_CONDITION_ROOT.glob('*_dynamic_soc.parquet'))
assert len(cmu_files) == 7, f'Expected seven CMU cell files, found {len(cmu_files)}'
display(pd.DataFrame({'path': [str(p) for p in cmu_files], 'size_mb': [round(p.stat().st_size / 1e6, 1) for p in cmu_files]}))
print('Panasonic MAT files:', len(list(PANASONIC_ROOT.rglob('*.mat'))))
print('Preflight passed. Set RUN_EXPERIMENT=True in the config cell to train.')

,path,size_mb
0,/content/battery-soc-transfer-benchmark/output...,21.5
1,/content/battery-soc-transfer-benchmark/output...,6.7
2,/content/battery-soc-transfer-benchmark/output...,4.0
3,/content/battery-soc-transfer-benchmark/output...,13.2
4,/content/battery-soc-transfer-benchmark/output...,9.3
5,/content/battery-soc-transfer-benchmark/output...,4.4
6,/content/battery-soc-transfer-benchmark/output...,6.1


Panasonic MAT files: 51
Preflight passed. Set RUN_EXPERIMENT=True in the config cell to train.


In [33]:
if RUN_EXPERIMENT:
    command = [
        sys.executable, str(REPO / 'scripts/benchmarks/run_bian_cmu.py'),
        '--panasonic-data-root', str(PANASONIC_ROOT),
        '--cmu-base-root', str(CMU_BASE_ROOT),
        '--cmu-condition-root', str(CMU_CONDITION_ROOT),
        '--cmu-manifest', str(CMU_MANIFEST),
        '--output-root', str(RESULT_ROOT),
        '--initial-soc', str(INITIAL_SOC),
        '--sequence-length', '20',
        '--source-epochs', str(setting['source_epochs']),
        '--transfer-epochs', str(setting['transfer_epochs']),
        '--repeats', str(setting['repeats']),
        '--batch-size', str(setting['batch_size']),
        '--transfer-lr', '1e-4',
        '--transfer-batch-size', '16',
        '--device', DEVICE,
    ]
    if setting['lightweight']:
        command.append('--lightweight')
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Dry run only. Set RUN_EXPERIMENT=True in the config cell when ready.')

Running: /usr/bin/python3 /content/battery-soc-transfer-benchmark/scripts/benchmarks/run_bian_cmu.py --panasonic-data-root /content/drive/MyDrive/battery_soc_benchmark/data/raw/Panasonic_18650PF --cmu-base-root /content/battery-soc-transfer-benchmark/outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1 --cmu-condition-root /content/battery-soc-transfer-benchmark/outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1_extension4 --cmu-manifest /content/battery-soc-transfer-benchmark/data/manifests/cmu_evtol/cmu_evtol_supervised_budget_manifest_v0_1.csv --output-root /content/drive/MyDrive/battery_soc_benchmark/outputs/benchmarks/panasonic_to_cmu_evtol/bian_2021/initial_soc_70_v2/formal --initial-soc 0.7 --sequence-length 20 --source-epochs 20 --transfer-epochs 5 --repeats 5 --batch-size 64 --transfer-lr 1e-4 --transfer-batch-size 16 --device cuda --lightweight


In [34]:
metrics_path = RESULT_ROOT / 'metrics.csv'
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics.pivot_table(index=['cell_id', 'initial_soc_percent'], columns='method', values=['overall_mae_percent', 'overall_rmse_percent'], aggfunc='mean').round(3))
    aggregate = metrics.groupby('method')[['overall_mae_percent', 'overall_rmse_percent']].mean().round(3)
    display(aggregate.style.set_caption('Core-seven mean (%)'))
    rmse = metrics.pivot_table(index='cell_id', columns='method', values='overall_rmse_percent', aggfunc='mean')
    display((rmse['Bian MDA'] - rmse['Source only']).rename('MDA_minus_source_RMSE_points').round(3).to_frame())
    print('Metadata:', RESULT_ROOT / 'run.json')
else:
    print('No result yet:', metrics_path)

overall_mae_percent              \
method                                 Bian MDA Source only   
cell_id initial_soc_percent                                   
VAH01   70                                3.265       6.651   
VAH11   70                                6.036       4.831   
VAH12   70                                6.952       7.183   
VAH15   70                                3.154       9.422   
VAH17   70                                3.013       7.919   
VAH27   70                                3.285       7.056   
VAH30   70                                3.024       9.943   

                            overall_rmse_percent              
method                                  Bian MDA Source only  
cell_id initial_soc_percent                                   
VAH01   70                                 4.859       7.377  
VAH11   70                                 7.493       6.414  
VAH12   70                                 8.599       8.404  
VAH15   70                                 4.721      10.067  
VAH17   70                                 4.626       8.596  
VAH27   70                                 4.743       7.701  
VAH30   70                                 4.392      10.558

,overall_mae_percent,overall_rmse_percent
method,,
Bian MDA,4.104000,5.633000
Source only,7.572000,8.445000


,MDA_minus_source_RMSE_points
cell_id,
VAH01,-2.517
VAH11,1.078
VAH12,0.195
VAH15,-5.346
VAH17,-3.970
VAH27,-2.958
VAH30,-6.166


Metadata: /content/drive/MyDrive/battery_soc_benchmark/outputs/benchmarks/panasonic_to_cmu_evtol/bian_2021/initial_soc_70_v2/formal/run.json
